In [1]:
import fastf1
import pandas as pd
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

/Users/chethanjujjavarapu/Desktop/GitHub/kyrios/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Load 2023 Japan Grand Prix for Understanding

In [2]:
# Enable caching to speed up data loading (optional but recommended)
# Cache directory will be created automatically in the scripts directory
if '__file__' in globals():
    # Running as a script
    file_path = os.path.abspath(__file__)
    scripts_dir = os.path.dirname(file_path)
    cache_dir = os.path.join(scripts_dir, 'cache')
else:
    # Running in IPython/interactive mode - use current working directory
    cache_dir = os.path.join(os.getcwd(), 'cache')

# Create cache directory if it doesn't exist
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

# Load a session (example: 2023 Bahrain Grand Prix, Race)
year = 2023
gp = 'Japan'
session_type = 'R'  # R = Race, Q = Qualifying, FP1/FP2/FP3 = Practice

session = fastf1.get_session(year, gp, session_type)
session.load()  # Load all available data for this session

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.076000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '44', '55', '63', '14', '31', '10', '40', '2

## Understanding session.race_control_messages

`session.race_control_messages` is a **DataFrame** containing all official race control messages issued during the session. These are communications from race control to teams and drivers about penalties, investigations, warnings, track conditions, and other critical race information.

### What it contains:
- Timestamped messages from race control
- Penalties (time penalties, grid penalties, etc.)
- Driver/team investigations
- Warnings and instructions
- Track condition announcements
- Safety car/VSC announcements

### Key Use Cases:
- Identifying penalties that affected race results
- Understanding race control decisions and their timing
- Analyzing driver behavior (investigations, warnings)
- Correlating race events with performance impacts
- Creating features for ML models (e.g., penalty flags)


In [4]:
# Basic info about the race_control_messages DataFrame
print("=== RACE CONTROL MESSAGES STRUCTURE ===")
print(f"Type: {type(session.race_control_messages)}")
print(f"Shape: {session.race_control_messages.shape}")
print(f"Number of messages: {len(session.race_control_messages)}")
print(f"\nColumns: {list(session.race_control_messages.columns)}")
print(f"\nIndex: {session.race_control_messages.index.name if session.race_control_messages.index.name else 'Default Index'}")
print(f"\nData types:\n{session.race_control_messages.dtypes}")


=== RACE CONTROL MESSAGES STRUCTURE ===
Type: <class 'pandas.core.frame.DataFrame'>
Shape: (71, 9)
Number of messages: 71

Columns: ['Time', 'Category', 'Message', 'Status', 'Flag', 'Scope', 'Sector', 'RacingNumber', 'Lap']

Index: Default Index

Data types:
Time            datetime64[ns]
Category                object
Message                 object
Status                  object
Flag                    object
Scope                   object
Sector                 float64
RacingNumber            object
Lap                      int64
dtype: object


In [5]:
# Display the race control messages data
print("=== RACE CONTROL MESSAGES PREVIEW ===")
print(f"Total messages: {len(session.race_control_messages)}")
session.race_control_messages


=== RACE CONTROL MESSAGES PREVIEW ===
Total messages: 71


,Time,Category,Message,Status,Flag,Scope,Sector,RacingNumber,Lap
0,2023-09-24 04:10:00,Flag,GREEN LIGHT - PIT EXIT OPEN,None,GREEN,Track,NaN,None,1
1,2023-09-24 04:20:00,Other,PIT EXIT CLOSED,None,None,None,NaN,None,1
2,2023-09-24 04:45:06,Other,RISK OF RAIN FOR F1 RACE IS 0%,None,None,None,NaN,None,1
3,2023-09-24 04:57:04,Drs,DRS DISABLED,DISABLED,None,None,NaN,None,1
4,2023-09-24 05:04:05,Flag,GREEN LIGHT - PIT EXIT OPEN,None,GREEN,Track,NaN,None,1
...,...,...,...,...,...,...,...,...,...
66,2023-09-24 06:33:17,Flag,WAVED BLUE FLAG FOR CAR 40 (LAW) TIMED AT 15:3...,None,BLUE,Driver,NaN,40,52
67,2023-09-24 06:35:04,Flag,CHEQUERED FLAG,None,CHEQUERED,Track,NaN,None,53
68,2023-09-24 06:36:14,Other,CAR 44 (HAM) TIME 1:37.353 DELETED - TRACK LIM...,None,None,None,NaN,None,53
69,2023-09-24 06:38:09,Flag,DOUBLE YELLOW IN TRACK SECTOR 4,None,DOUBLE YELLOW,Sector,4.0,None,53


In [6]:
# Display all messages with better formatting
print("=== ALL RACE CONTROL MESSAGES ===")
print(f"Total messages: {len(session.race_control_messages)}\n")
for idx, row in session.race_control_messages.iterrows():
    time_str = str(row['Time'])
    message = row['Message']
    print(f"[{time_str}] {message}\n")


=== ALL RACE CONTROL MESSAGES ===
Total messages: 71

[2023-09-24 04:10:00] GREEN LIGHT - PIT EXIT OPEN

[2023-09-24 04:20:00] PIT EXIT CLOSED

[2023-09-24 04:45:06] RISK OF RAIN FOR F1 RACE IS 0%

[2023-09-24 04:57:04] DRS DISABLED

[2023-09-24 05:04:05] GREEN LIGHT - PIT EXIT OPEN

[2023-09-24 05:04:32] YELLOW IN TRACK SECTOR 4

[2023-09-24 05:04:38] CLEAR IN TRACK SECTOR 4

[2023-09-24 05:05:18] SAFETY CAR DEPLOYED

[2023-09-24 05:06:30] DOUBLE YELLOW IN TRACK SECTOR 2

[2023-09-24 05:08:44] LAP 1 TURN 1 NOTED

[2023-09-24 05:11:45] INCIDENT INVOLVING CAR 11 (PER) NOTED - SAFETY CAR INFRINGEMENT

[2023-09-24 05:12:36] CLEAR IN TRACK SECTOR 2

[2023-09-24 05:12:41] SAFETY CAR IN THIS LAP

[2023-09-24 05:13:56] TRACK CLEAR

[2023-09-24 05:16:02] CAR 77 (BOT) OFF TRACK AND CONTINUED AT TURN 11

[2023-09-24 05:16:55] TRACK SURFACE SLIPPERY IN TRACK SECTOR 11

[2023-09-24 05:17:07] TRACK SURFACE SLIPPERY IN TRACK SECTOR 12

[2023-09-24 05:17:13] CLEAR IN TRACK SECTOR 12

[2023-09-24 05:1

### Column Descriptions:

1. **Time** - Timestamp when the race control message was issued (relative to session start)
   - Format: Timedelta (e.g., "0 days 00:15:32.123000")
   - Use this to correlate messages with lap times and race events

2. **Message** - The full text of the race control message
   - Contains detailed information about the event
   - Typically includes driver/team names, penalty types, amounts, reasons
   - Examples:
     - "Car 44 given 5 second time penalty for causing a collision"
     - "Investigation - Car 33 - Incident involving Car 1"
     - "DRS enabled"
     - "Virtual Safety Car deployed"
     - "All cars must use intermediate or wet weather tyres"

### Common Message Types:

- **Penalties**: Time penalties, grid penalties, drive-through penalties
- **Investigations**: Driver investigations for incidents
- **Warnings**: Driver warnings for various infractions
- **Track Conditions**: DRS enable/disable, tire restrictions
- **Safety**: Safety car, VSC deployments, flag conditions
- **Instructions**: General instructions to drivers/teams

### Important Notes:

- **Timestamps**: All times are relative to session start (same reference as other session data)
- **Variable length**: Number of messages varies significantly between sessions
- **Text data**: Messages are free-form text, requiring parsing for structured analysis
- **Event correlation**: Messages often correlate with track_status changes
- **Driver identification**: Driver numbers are often mentioned in messages (e.g., "Car 44")


In [7]:
# Analyze message patterns
print("=== MESSAGE ANALYSIS ===")
print(f"Time range:")
print(f"  First message: {session.race_control_messages['Time'].min()}")
print(f"  Last message: {session.race_control_messages['Time'].max()}")

# Count messages by type (keyword-based)
messages = session.race_control_messages['Message'].str.lower()

print(f"\n=== MESSAGE TYPE COUNTS (keyword-based) ===")
keyword_counts = {
    'Penalty': messages.str.contains('penalty', na=False).sum(),
    'Investigation': messages.str.contains('investigation', na=False).sum(),
    'Warning': messages.str.contains('warning', na=False).sum(),
    'Safety Car': messages.str.contains('safety car', na=False).sum(),
    'VSC': messages.str.contains('virtual safety car|vsc', na=False).sum(),
    'DRS': messages.str.contains('drs', na=False).sum(),
    'Flag': messages.str.contains('flag', na=False).sum(),
    'Tyre/Tire': messages.str.contains('tyre|tire', na=False).sum(),
}

for keyword, count in keyword_counts.items():
    if count > 0:
        print(f"{keyword}: {count}")

# Extract driver numbers mentioned in messages (if any)
import re
driver_numbers = []
for msg in session.race_control_messages['Message']:
    # Look for patterns like "Car 44" or "Driver 44"
    matches = re.findall(r'(?:Car|Driver)\s+(\d+)', msg, re.IGNORECASE)
    driver_numbers.extend(matches)

if driver_numbers:
    from collections import Counter
    driver_mentions = Counter(driver_numbers)
    print(f"\n=== DRIVER MENTIONS IN MESSAGES ===")
    for driver, count in driver_mentions.most_common():
        print(f"Driver {driver}: {count} mention(s)")
else:
    print("\nNo driver numbers found in messages")


=== MESSAGE ANALYSIS ===
Time range:
  First message: 2023-09-24 04:10:00
  Last message: 2023-09-24 06:38:34

=== MESSAGE TYPE COUNTS (keyword-based) ===
Penalty: 3
Investigation: 6
Safety Car: 7
VSC: 2
DRS: 3
Flag: 5

=== DRIVER MENTIONS IN MESSAGES ===
Driver 11: 5 mention(s)
Driver 44: 3 mention(s)
Driver 63: 3 mention(s)
Driver 81: 2 mention(s)
Driver 22: 2 mention(s)
Driver 77: 1 mention(s)
Driver 2: 1 mention(s)
Driver 14: 1 mention(s)
Driver 20: 1 mention(s)
Driver 23: 1 mention(s)
Driver 4: 1 mention(s)
Driver 27: 1 mention(s)
Driver 16: 1 mention(s)
Driver 24: 1 mention(s)
Driver 40: 1 mention(s)


## Practical Usage Examples

### 1. Extract penalties for specific drivers
```python
# Find all messages containing penalties for a specific driver
driver_num = '44'
penalty_messages = session.race_control_messages[
    session.race_control_messages['Message'].str.contains(f'Car {driver_num}.*penalty', case=False, na=False)
]
```

### 2. Count penalties per driver
```python
import re
from collections import defaultdict

penalties_by_driver = defaultdict(int)
for msg in session.race_control_messages['Message']:
    if 'penalty' in msg.lower():
        matches = re.findall(r'(?:Car|Driver)\s+(\d+)', msg, re.IGNORECASE)
        for driver in matches:
            penalties_by_driver[driver] += 1
```

### 3. Filter messages by type
```python
# Get all penalty messages
penalties = session.race_control_messages[
    session.race_control_messages['Message'].str.contains('penalty', case=False, na=False)
]

# Get all investigation messages
investigations = session.race_control_messages[
    session.race_control_messages['Message'].str.contains('investigation', case=False, na=False)
]
```

### 4. Using race_control_messages as features for ML models
- **Binary flags**: Did driver X receive a penalty? (0/1 per driver)
- **Penalty count**: Number of penalties per driver
- **Investigation count**: Number of investigations per driver
- **Warning count**: Number of warnings per driver
- **Message count**: Total number of race control messages (indicates eventful race)

### 5. Correlate messages with lap times
```python
# Messages affect driver performance - merge with laps data
# Use Time column to match with LapStartTime
# Create features like "penalty_received_before_lap" or "investigation_active_during_lap"
```

### 6. Extract penalty amounts and types
```python
import re

# Extract time penalties (e.g., "5 second penalty")
for msg in session.race_control_messages['Message']:
    time_penalties = re.findall(r'(\d+)\s*second.*penalty', msg, re.IGNORECASE)
    if time_penalties:
        print(f"Time penalty: {time_penalties[0]} seconds")
```

### 7. Link with track_status
```python
# Race control messages often precede track status changes
# (e.g., "Safety Car deployed" message → track_status changes to SC)
# Can be used to understand the sequence of events
```

### Important Considerations:
- **Text parsing**: Messages are free-form text, so parsing is needed for structured features
- **Driver matching**: Need to extract driver numbers from messages and match with driver data
- **Time correlation**: Messages can be merged with laps data using Time/LapStartTime
- **Sparse data**: Many sessions have few or no messages, so features may be mostly zeros
- **Causal direction**: Messages often reflect events that already happened, so timing matters
